In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_chroma import Chroma
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

In [ ]:
#### Approach 1: Using Langchain with Ollama and Chroma

# Load PDF
loader = PyPDFLoader("Drivers-Handbook.pdf")
documents = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)

# Embeddings
embeddings = OllamaEmbeddings(model="llama3.1:8b")

# Vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# LLM
llm = OllamaLLM(model="llama3.2", temperature=0.3)

# QA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True
)

# Query
result = qa_chain.invoke({"query": "What is the speed limit in Germany?"})
print(result['result'])

InvalidArgumentError: Collection expecting embedding with dimension of 3072, got 4096

In [15]:
# Query
result = qa_chain.invoke({"query": "give some traffic rules mentioned?"})
print(result['result'])

Here are some traffic rules mentioned:

1. Drivers must use discretion when choosing a speed and adjust it to meet road, traffic, and weather conditions.
2. There is a maximum speed limit of 25 kph for motor vehicles using certain roads (e.g. tractor-pulled farm vehicles or Mofas).
3. Signs with specific conditions must be obeyed under those conditions (e.g. Nässe sign 1052-36 requires observing the speed limit only when the road is wet).
4. Drivers should avoid noise or exhaust fumes produced by heavy traffic.
5. There are rules against certain "avoidable circumstances" that drivers can control, such as squealing tires in residential areas, racing the motor, honking the horn, and playing loud music.


In [ ]:
###### Approach 2: Using LangChain's new structure with LCEL

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Load and process PDF
loader = PyPDFLoader("Drivers-Handbook.pdf")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

# Create vector store
embeddings = OllamaEmbeddings(model="llama3.1:8b")
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Create prompt
template = """Answer the question based only on the following context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# Create LLM
llm = OllamaLLM(model="llama3.2", temperature=0.3)

# Create chain using LCEL
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Query
question = "What is the speed limit in German cities?"
answer = chain.invoke(question)
print(answer)

KeyboardInterrupt: 

In [ ]:
# traffic_rules_rag.py
# Approach 3: Latest LangChain implementation

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_chroma import Chroma
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_core.prompts import PromptTemplate
import os

class TrafficRulesRAG:
    def __init__(self, pdf_path: str, model_name: str = "llama3.2"):
        self.pdf_path = pdf_path
        self.model_name = model_name
        self.vectorstore = None
        self.qa_chain = None
        
    def setup(self):
        """Setup the RAG system"""
        
        # Check if database exists
        if os.path.exists("./chroma_db"):
            print("Loading existing vector database...")
            embeddings = OllamaEmbeddings(model=self.model_name)
            self.vectorstore = Chroma(
                persist_directory="./chroma_db",
                embedding_function=embeddings
            )
            print("✅ Database loaded")
        else:
            print("Creating new vector database...")
            self._create_vectorstore()
        
        # Create QA chain
        self._create_qa_chain()
        
    def _create_vectorstore(self):
        """Create vector store from PDF"""
        
        # Load PDF
        print("Loading PDF...")
        loader = PyPDFLoader(self.pdf_path)
        documents = loader.load()
        print(f"✅ Loaded {len(documents)} pages")
        
        # Split documents
        print("Splitting into chunks...")
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
        chunks = text_splitter.split_documents(documents)
        print(f"✅ Created {len(chunks)} chunks")
        
        # Create embeddings
        print("Creating embeddings (this may take a few minutes)...")
        embeddings = OllamaEmbeddings(model=self.model_name)
        
        # Create vector store
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory="./chroma_db"
        )
        print("✅ Vector database created")
    
    def _create_qa_chain(self):
        """Create the QA chain with custom prompt"""
        
        # Custom prompt template
        template = """You are an expert on German traffic rules. Use the following context to answer the question.
        If you don't know the answer, just say you don't know. Don't make up an answer.
        
        Context: {context}
        
        Question: {question}
        
        Answer: """
        
        prompt = PromptTemplate(
            template=template,
            input_variables=["context", "question"]
        )
        
        # Create LLM
        llm = OllamaLLM(
            model=self.model_name,
            temperature=0.3
        )
        
        # Create chain
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=self.vectorstore.as_retriever(
                search_type="similarity",
                search_kwargs={"k": 3}
            ),
            return_source_documents=True,
            chain_type_kwargs={"prompt": prompt}
        )
        
        print("✅ QA chain ready")
    
    def ask(self, question: str):
        """Ask a question"""
        
        if not self.qa_chain:
            raise ValueError("QA chain not initialized. Run setup() first.")
        
        print(f"\n{'='*70}")
        print(f"Question: {question}")
        print("="*70)
        
        result = self.qa_chain.invoke({"query": question})
        
        print("\n📝 Answer:")
        print("-"*70)
        print(result['result'])
        
        print("\n📚 Sources:")
        print("-"*70)
        for i, doc in enumerate(result['source_documents'], 1):
            page = doc.metadata.get('page', 'N/A')
            preview = doc.page_content[:150].replace('\n', ' ')
            print(f"{i}. Page {page}: {preview}...")
        
        return result


    

In [ ]:

# Initialize RAG system
rag = TrafficRulesRAG(
    pdf_path="Drivers-Handbook.pdf",
    model_name="llama3.2"
)

# Setup (one time)
rag.setup()

print("\n" + "="*70)
print("🚦 Traffic Rules Assistant Ready!")
print("="*70)

# Ask questions
rag.ask("What is the speed limit in urban areas in Germany?")
rag.ask("What does a red triangle sign mean?")
rag.ask("Where am I not allowed to park?")

# Interactive mode
print("\n" + "="*70)
print("Interactive Mode - Type 'quit' to exit")
print("="*70)

while True:
    question = input("\n🚦 Your question: ")
    if question.lower() in ['quit', 'exit', 'q']:
        print("\nGoodbye! 👋")
        break
    if question.strip():
        rag.ask(question)

Loading existing vector database...
✅ Database loaded
✅ QA chain ready

🚦 Traffic Rules Assistant Ready!

Question: What is the speed limit in urban areas in Germany?

📝 Answer:
----------------------------------------------------------------------
I don't know the answer to that question. The provided text only includes information about specific traffic signs and does not mention speed limits.

📚 Sources:
----------------------------------------------------------------------
1. Page 70: sign must be obeyed only  when the road is wet.  No Stopping on Shoulder.  Prohibits stopping on the  road or shoulder where this  sign is posted.  NA...
2. Page 70: sign must be obeyed only  when the road is wet.  No Stopping on Shoulder.  Prohibits stopping on the  road or shoulder where this  sign is posted.  NA...
3. Page 70: sign must be obeyed only  when the road is wet.  No Stopping on Shoulder.  Prohibits stopping on the  road or shoulder where this  sign is posted.  NA...

Question: What does